# Thành viên 2 – Feature Engineering
GROUP BY & Aggregation trên các bảng phụ: bureau, previous_application, installments, credit_card, POS_CASH.

In [ ]:
import sys
sys.path.append('../src')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder \
    .appName('HomeCreditFeatures') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

## 1. Load bảng phụ

In [ ]:
CLEANED = '../output/cleaned_data'
DATA    = '../data'

bureau          = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/bureau.csv')
bureau_balance  = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/bureau_balance.csv')
prev            = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/previous_application.csv')
inst            = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/installments_payments.csv')
cc              = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/credit_card_balance.csv')
pos             = spark.read.option('header',True).option('inferSchema',True).csv(f'{DATA}/POS_CASH_balance.csv')

for name, df in [('bureau',bureau),('prev_app',prev),('installments',inst),('credit_card',cc),('pos_cash',pos)]:
    print(f'{name}: {df.count():,} rows')

## 2. Aggregate Bureau

In [ ]:
from feature_engineering import aggregate_bureau, aggregate_bureau_balance

agg_bb = aggregate_bureau_balance(bureau_balance)
print('Bureau balance aggregated:'); agg_bb.show(3)

agg_bureau = aggregate_bureau(bureau)
print('Bureau aggregated:'); agg_bureau.show(3)

## 3. Aggregate Previous Application

In [ ]:
from feature_engineering import aggregate_previous_application

agg_prev = aggregate_previous_application(prev)
agg_prev.show(3)

# Visualise tỉ lệ approved vs refused
status_pd = prev.groupBy('NAME_CONTRACT_STATUS').count().toPandas()
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(status_pd['NAME_CONTRACT_STATUS'], status_pd['count'], color='teal')
ax.set_title('Previous Application – Contract Status')
ax.set_xlabel('Status'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/EDA_visualizations/prev_app_status.png', dpi=150)
plt.show()

## 4. Aggregate Installments, Credit Card, POS CASH

In [ ]:
from feature_engineering import aggregate_installments, aggregate_credit_card, aggregate_pos_cash

agg_inst = aggregate_installments(inst)
agg_cc   = aggregate_credit_card(cc)
agg_pos  = aggregate_pos_cash(pos)

for name, df in [('installments',agg_inst),('credit_card',agg_cc),('pos_cash',agg_pos)]:
    print(f'\n{name} ({df.count():,} rows):')
    df.show(3)

In [ ]:
spark.stop()